# 1. 라이브러리 설치
* 가상환경 설정 후 requirements.txt 설치 진행
  `pip install -r requirements.txt`

# 2. 필요한 라이브러리 import

In [1]:
from gensim.models import Doc2Vec
from kiwipiepy import Kiwi  # 형태소 분석기

# 3. 한국어 문장(Document)을 임베딩 벡터로 변환

In [3]:
from gensim.models.doc2vec import TaggedDocument

docs=[
    '자바스크립트는 웹사이트에 동적인 기능을 추가하는 핵심 프로그래밍 언어입니다',
    '일본 도쿄는 현대적인 도심과 전통 문화가 공존하는 인기 여행지입니다.',
    '파이썬은 데이터 분석과 인공지능 개발에 널리 사용되는 프로그래밍 언어입니다.',
    '김치찌개는 매운맛을 즐기는 한국인들에게 사랑받는 대표적인 국물 요리입니다.',
    '스페인 바르셀로나는 건축가 가우디의 예술 작품과 맛있는 해산물 요리로 유명합니다'
]

# 형태소 분석기
kiwi = Kiwi()

# 유지할 품사 태그: NNG(일반명사), NNP(고유명사), NNB(의존명사), VV(동사), VA(형용사)
keep_tags = {'NNG', 'NNP', 'NNB', 'VV', 'VA'}

documents = []
for i, doc in enumerate(docs):
    # TaggedDocument: 문서에 태그를 붙여줌.
    # TaggedDocument(tags=[문서번호], words=[형태소 단어들])
    words = [token.form for token in kiwi.tokenize(doc)  # 품사 태그 필터링
             if token.tag in keep_tags]
    documents.append(TaggedDocument(tags=[i], words=words))

'''
Doc2Vec은 단어 벡터 + 문서 벡터를 함께 학습

TaggedDocument(tags=[i], words=words)
#              ^^^^^^^^  ^^^^^^^^^^
#              문서 ID    해당 문서의 단어 목록
#              (0, 1, 2...)
'''
print('문서의 수 :', len(documents))
print(documents)  # 토큰 확인


문서의 수 : 5
[TaggedDocument(words=['자바스크립트', '웹', '사이트', '동적', '기능', '추가', '핵심', '프로그래밍', '언어'], tags=[0]), TaggedDocument(words=['일본', '도쿄', '현대', '도심', '전통', '문화', '공존', '인기', '여행지'], tags=[1]), TaggedDocument(words=['파이썬', '데이터', '분석', '인공', '지능', '개발', '사용', '프로그래밍', '언어'], tags=[2]), TaggedDocument(words=['김치찌개', '맛', '즐기', '한국인', '사랑', '대표', '국물', '요리'], tags=[3]), TaggedDocument(words=['스페인', '바르셀로나', '건축가', '가우디', '예술', '작품', '맛있', '해산물', '요리'], tags=[4])]


In [4]:
# 여기서 학습률이란, 모델이 학습을 통해 가중치를 업데이트하는 정도를 의미
# 학습률이 높을수록 가중치 업데이트가 크게 이루어지며, 낮을수록 작게 이루어짐.
model = Doc2Vec(
    vector_size=300,    # 벡터의 차원
    min_count=1,        # 단어 최소 빈도수
    alpha=1.0,            # 학습률
    min_alpha=0.025,    # 학습률 초기값
    window=4            # 문맥의 크기
  )

In [5]:
# Vocabulary 빌드
# 모델이 학습 할 문서 빌드
model.build_vocab(documents)

# Doc2Vec 학습
# model.corpus_count: 학습 할 문서의 총 개수
# epochs: 학습 할 에포크 수
model.train(documents, total_examples=model.corpus_count, epochs=20)

In [6]:
# 첫번째 문장과 가장 유사한 문장은 1번 문장?
result = model.dv.most_similar(0)
'''
출력 결과
[
    (2, 0.9801342487335205), 
    (3, 0.8567307591438293), 
    (1, 0.7260693311691284), 
    (4, 0.7010143399238586)
]
'''
print(result)

[(2, 0.9801342487335205), (3, 0.8567306399345398), (1, 0.7260693311691284), (4, 0.7010143399238586)]
